# Titan Embedding Model Gateway Testing

This notebook tests Amazon Titan embedding models through the Bedrock Proxy Gateway.

**Important**: Embedding models only support `invoke_model` API. They do not support:
- `converse` (conversational API)
- `converse_stream` (streaming conversational API)
- `invoke_model_with_response_stream` (streaming API)

Embedding models return vector representations, not text responses.

## Initialization and Configuration

In [ ]:
import json

import boto3
import requests
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError


import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment-specific .env file (e.g., .env.dev, .env.test)
ENVIRONMENT = os.getenv("ENVIRONMENT", "dev")
env_file = Path(__file__).parent / f".env.{ENVIRONMENT}" if "__file__" in dir() else Path(f".env.{ENVIRONMENT}")
load_dotenv(env_file)

# Configuration
TITAN_EMBEDDING_MODEL = "amazon.titan-embed-text-v1"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_URL = os.getenv("GATEWAY_API_URL", "https://your-gateway-url.com")
SECRET_ID = os.getenv("GATEWAY_SECRET_ID", "bedrock-gateway-dev-oauth-credentials")

# Extract the secret value from Secret Manager

session = boto3.Session(profile_name=os.getenv("AWS_PROFILE"), region_name=AWS_REGION)
JSON_SECRET = json.loads(session.client('secretsmanager').get_secret_value(
    SecretId=SECRET_ID)['SecretString'])
CLIENT_ID = JSON_SECRET['client_id']
CLIENT_SECRET = JSON_SECRET['client_secret']
TOKEN_URL = JSON_SECRET['token_url']

## Token Generation

In [ ]:
def generate_token():
    """Generate access token for API authentication"""
    payload = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "client_credentials",
        "audience": "bedrockproxygateway",
        "scope": "bedrockproxygateway:invoke"
    }

    response = requests.post(TOKEN_URL, data=payload)
    response.raise_for_status()
    token = response.json()['access_token']
    return f"Bearer {token}"

def create_bedrock_client(token):
    """Create and configure Bedrock runtime client"""
    bedrock = boto3.client(
        'bedrock-runtime',
        region_name=AWS_REGION,
        endpoint_url=API_URL,
        config=Config(signature_version=UNSIGNED, retries={'max_attempts': 0}),
        verify=False
    )

    def add_api_token(request, **kwargs):
        request.headers.add_header('Authorization', token)
        return request
    
    bedrock.meta.events.register('before-sign.bedrock-runtime.*', add_api_token)
    return bedrock

## Bedrock Runtime Client Setup

In [ ]:
# Generate token
TOKEN = generate_token()
print(f"Token generated: {TOKEN[:20]}...")

# Create Bedrock client
bedrock = create_bedrock_client(TOKEN)
print("Bedrock client configured successfully")

## Test 1: InvokeModel (Supported API)

This is the correct and only supported API for embedding models.

In [ ]:
# Text to embed
text_to_embed = "AWS Bedrock is a fully managed service that offers foundation models from leading AI companies."

# Prepare request for Titan embedding model (v1 only supports inputText)
titan_request = {
    "inputText": text_to_embed
}

try:
    print(f"Testing invoke_model with {TITAN_EMBEDDING_MODEL}...\n")
    # Call Bedrock invoke model API for embeddings
    response = bedrock.invoke_model(
        modelId=TITAN_EMBEDDING_MODEL,
        body=json.dumps(titan_request),
        contentType="application/json",
        accept="application/json"
    )
    # Parse response
    response_body = json.loads(response["body"].read())
    embeddings = response_body["embedding"]
    print("✅ SUCCESS: invoke_model works with embedding models")
    print(f"\nModel: {TITAN_EMBEDDING_MODEL}")
    print(f"Input text: {text_to_embed}")
    print(f"Embedding dimensions: {len(embeddings)}")
    print(f"First 10 embedding values: {embeddings[:10]}")
    print(f"Input token count: {response_body.get('inputTextTokenCount', 'N/A')}")
    # Print response headers
    print("\nResponse Headers:")
    for key, value in response.get('ResponseMetadata', {}).get('HTTPHeaders', {}).items():
        print(f"  {key}: {value}")
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"❌ ERROR: Can't invoke '{TITAN_EMBEDDING_MODEL}'.")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"❌ ERROR: Unexpected error. Reason: {e}")

## Test 2-4: Unsupported APIs (Expected to Fail)

The following tests demonstrate that embedding models do NOT support conversational or streaming APIs.
These are expected to fail with appropriate error messages.

### Test 2: Converse API (Not Supported)

In [ ]:
# Attempt to use converse API with embedding model
conversation = [
    {
        "role": "user",
        "content": [{"text": text_to_embed}],
    }
]

try:
    print(f"Testing converse with {TITAN_EMBEDDING_MODEL}...\n")
    response = bedrock.converse(
        modelId=TITAN_EMBEDDING_MODEL,
        messages=conversation
    )
    print("⚠️ UNEXPECTED: converse API succeeded (this should not happen)")
    print(response)
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"❌ EXPECTED FAILURE: converse API not supported for embedding models")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"❌ EXPECTED FAILURE: {e}")

### Test 3: Converse Stream API (Not Supported)

In [ ]:
# Attempt to use converse_stream API with embedding model
try:
    print(f"Testing converse_stream with {TITAN_EMBEDDING_MODEL}...\n")
    streaming_response = bedrock.converse_stream(
        modelId=TITAN_EMBEDDING_MODEL,
        messages=conversation
    )
    for chunk in streaming_response["stream"]:
        print(chunk)
    print("⚠️ UNEXPECTED: converse_stream API succeeded (this should not happen)")
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"❌ EXPECTED FAILURE: converse_stream API not supported for embedding models")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"❌ EXPECTED FAILURE: {e}")

### Test 4: InvokeModelWithResponseStream API (Not Supported)

In [ ]:
# Attempt to use invoke_model_with_response_stream API with embedding model
try:
    print(f"Testing invoke_model_with_response_stream with {TITAN_EMBEDDING_MODEL}...\n")
    streaming_response = bedrock.invoke_model_with_response_stream(
        modelId=TITAN_EMBEDDING_MODEL,
        body=json.dumps(titan_request)
    )
    for event in streaming_response["body"]:
        print(event)
    print("⚠️ UNEXPECTED: invoke_model_with_response_stream API succeeded (this should not happen)")
except ClientError as e:
    error_code = e.response.get('Error', {}).get('Code', 'Unknown')
    error_message = e.response.get('Error', {}).get('Message', str(e))
    status_code = e.response.get('ResponseMetadata', {}).get('HTTPStatusCode', 'Unknown')
    print(f"❌ EXPECTED FAILURE: invoke_model_with_response_stream API not supported for embedding models")
    print(f"Status Code: {status_code}")
    print(f"Error Code: {error_code}")
    print(f"Error Message: {error_message}")
except Exception as e:
    print(f"❌ EXPECTED FAILURE: {e}")

## Additional Tests: Multiple Text Embeddings

In [ ]:
# Test with multiple texts
texts = [
    "Amazon Web Services provides cloud computing services.",
    "Machine learning enables computers to learn from data.",
    "Natural language processing helps computers understand human language."
]

print("Testing multiple text embeddings...\n")

embeddings_list = []

for idx, text in enumerate(texts, 1):
    try:
        request = {
            "inputText": text
        }
        response = bedrock.invoke_model(
            modelId=TITAN_EMBEDDING_MODEL,
            body=json.dumps(request)
        )
        response_body = json.loads(response["body"].read())
        embeddings = response_body["embedding"]
        embeddings_list.append(embeddings)
        print(f"✅ Text {idx}: {text[:50]}...")
        print(f"   Dimensions: {len(embeddings)}, Tokens: {response_body.get('inputTextTokenCount', 'N/A')}\n")
    except Exception as e:
        print(f"❌ Text {idx} failed: {e}\n")

print(f"Successfully generated {len(embeddings_list)} embeddings")